## Lecture et écriture de données avec les DataFrames

### Objéctifs

- Lire des données externes dans un Dataframe
- Créer et inspécter les schémas de Dataframes
- Ecrire les contenus d'un Dataframe vers un cible (fichier, table, ...)

%md
Dans ce notebook il y a un paramètre `full_path` en haut du notebook

![](./Resources/NB02/img_parameters.png)

La cellule dessous permet de créer ce paramètre si il n'existe pas, il y aura une valeur par défaut vers l'emplacement du fichier MOCK_DATA.csv : `/Workspace/Users/{username}/databricks-training/Spark Developer/Introduction a Spark/Resources/NB02/MOCK_DATA.csv`

Comme Spark ne peut pas lire de fichier directement dans le système de fichier du Workspace comme on l'as fait avec Pands, il faut : 
- Créer un volume (dans un Catalog/schema)
- Placer le fichier dans le volume

Les deux prochaines cellules vont paramètrer les différentes variables nécéssaires pour le `Setup`. 

Par défaut un volume sera créer dans le catalog : **workspace.default** (spark_training) mais on peut modifier les widgets (directement dans le code ou si la cellules à déjà été exécuté en haut du notebook).

In [0]:
# Passe la variable 'full_path' avec le chemin vers le fichier CSV MOCK_DATA.csv au notebook NB02/Setup
import os

full_path = os.getcwd() + "/Resources/NB02/MOCK_DATA.csv"

dbutils.widgets.text("full_path", full_path)
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "default")
dbutils.widgets.text("volume", "spark_training")

In [0]:
%run "./Resources/NB02/Setup"

## Lecture d'un fichier CSV dans un Dataframe

Dans les cellules précédentes on à donc placer un fichier dans un volume Databricks.

L'objéctif est maintenant de lire ce fichier dans un Dataframe


In [0]:
# la variable path à été initialisé dans le notebook de setup dans Resources/NB02 c'est le chemin vers l'emplacement du fichier dans le volume :
# par défaut /Volumes/workspace/default/spark_training/MOCK_DATA.csv
# si on ne met pas l'option header à true les colonnes seront nommées _c0, _c1, etc... et devront être renommées manuellement
mock_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(path)

display(mock_df)

In [0]:
# montre le schéma déduit pour mock_df
mock_df.printSchema()

### Fonction display()

La fonction `display()` dans Databricks permet d'afficher visuellement le contenu d'un DataFrame, d'une table ou d'un résultat de requête SQL. Elle offre une interface interactive pour explorer les données : tri, filtrage, recherche, visualisation graphique (histogrammes, courbes, cartes, etc.), et export des résultats. On peut ainsi rapidement analyser et manipuler les données directement dans le notebook sans écrire de code supplémentaire pour la visualisation.

In [0]:
display(mock_df)

### Définition explicite d'un schéma

On va lire le même fichier dans un dataframe en définissant explicitement le schéma : 

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType

# définition du schéma
struct_schema = StructType([
    StructField("date", DateType(), True),
    StructField("mobile_device_brand", StringType(), True),
    StructField("mobile_device_os", StringType(), True),
    StructField("credit_card_type", StringType(), True),
    StructField("currency", StringType(), True)
])

# lecture du fichier CSV dans un dataframe avec le StructType schéma
mock_structtype_df = spark.read.format("csv") \
    .option("header", "true") \
    .schema(struct_schema) \
    .load(path)

display(mock_structtype_df)


### Schéma DDL

Le schéma DDL (Data Definition Language) est une façon de décrire la structure des données d'un DataFrame sous forme de chaîne de caractères. Il précise le nom des colonnes, leur type (string, integer, boolean, etc.) et éventuellement si elles peuvent contenir des valeurs nulles. Exemple : `id INT, name STRING, age INT`. Ce format est utilisé pour définir explicitement le schéma lors de la lecture de fichiers, garantissant une interprétation correcte des types de données.

Exemple de schéma DDL :  
```python
schema = """
  id INT, 
  name STRING, 
  age INT, 
  email STRING, 
  is_active BOOLEAN
"""
```

In [0]:

ddl_schema = """
    date DATE, 
    mobile_device_brand STRING, 
    mobile_device_os STRING, 
    credit_card_type STRING, 
    currency STRING
"""

mock_ddl_df = spark.read.format("csv") \
    .option("header", "true") \
    .schema(ddl_schema) \
    .load(path)

display(mock_ddl_df)

In [0]:
mock_ddl_df.printSchema()

## Ecriture du contenu depuis un Dataframe

### Vers un fichier

On peut écrire le contenu dans un fichier vers le système de fichier en utilisant les méthodes `write` et `save` : 

In [0]:
# défition du chemin de destination
# ce code utilise les valeurs des widgets pour le catalog, schema et volume définis dans le notebook de setup
parquet_output = f"/Volumes/{catalog}/{schema}/{volume}/MOCK_DATA_PARQUET"
print(parquet_output)

In [0]:
# Ecriture dans un fichier Parquet sur le volume
mock_ddl_df.write.format("parquet") \
    .mode("overwrite") \
    .save(parquet_output)

In [0]:
display(dbutils.fs.ls(parquet_output))

### Vers une Table

On peut écrire le contenu dans une table définie dans la Unity Catalog de Databricks :

In [0]:
mock_ddl_df.write.saveAsTable("mock_ddl_df_table")

In [0]:
# on peut également utiliser la méthode : 
mock_ddl_df.writeTo(
    f"{catalog}.{schema}.mock_ddl_df_table"
).createOrReplace()
# il y a les options pour partionner, ajouter, écrire par dessus, ...

In [0]:
%sql
-- Maintenant on peut lire la table
SELECT * FROM mock_ddl_df_table;